![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 24 Live Demo: Building Generative Models from Scratch

Today we are building three generative models from scratch on real images with real captions, then conditioning generation on text using CLIP. By the end you will see exactly how Stable Diffusion works -- just at a smaller scale.

**What we will build:**
1. Convolutional VAE (encoder/decoder with conv layers)
2. DCGAN (transposed convolutions to go from noise vector to image)
3. U-Net Diffusion Model (the architecture behind Stable Diffusion)
4. CLIP-Conditioned Diffusion (type text, get matching image)

**Dataset:** Flickr30k -- real photographs with human-written captions, used to train models like CLIP and BLIP.

**Runtime:** ~15 minutes total on T4 GPU

In [ ]:
!pip install datasets open_clip_torch huggingface_hub -q

In [ ]:
import torch                                    # core tensor library
import torch.nn as nn                           # neural network layers (Conv2d, Linear, etc.)
import torch.nn.functional as F                 # functions (relu, mse_loss, etc.)
import torch.optim as optim                     # optimizers (Adam)
from torch.utils.data import DataLoader, Dataset  # data loading
import torchvision                              # image utilities
import torchvision.transforms as T              # image transforms (resize, normalize)
import matplotlib.pyplot as plt                 # plotting
import numpy as np                              # numerical ops
from tqdm import tqdm                           # progress bars
from PIL import Image                           # image loading

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Image size for all models -- 64x64 is the sweet spot:
# big enough to see real structure, small enough to train fast
IMG_SIZE = 64

In [ ]:
# ============================================================
# HELPER: Display a grid of generated images
# ============================================================
# We will call this after every model to see what it produces.

def show_images(images, title="Generated Images", nrow=8):
    """Display a batch of images in a grid.
    images: tensor (N, 3, 64, 64) with values in [0,1] or [-1,1]
    """
    images = images.detach().cpu().float()         # move to CPU
    images = (images - images.min()) / (images.max() - images.min() + 1e-8)  # normalize to [0,1]
    grid = torchvision.utils.make_grid(images, nrow=nrow, padding=2)  # arrange in grid
    plt.figure(figsize=(14, 7))
    plt.imshow(grid.permute(1, 2, 0).numpy())     # CHW -> HWC for matplotlib
    plt.title(title, fontsize=14)
    plt.axis('off')
    plt.tight_layout()
    plt.show()


def plot_loss(losses, title="Training Loss"):
    """Plot a loss curve."""
    plt.figure(figsize=(8, 3))
    plt.plot(losses, color='steelblue', alpha=0.7)
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

---
## Part 0: Getting the Data (Flickr30k from HuggingFace)

Before we touch any model, we need data. We are using **Flickr30k** -- a real dataset of 31,000 photographs, each with 5 human-written English captions. This is one of the standard benchmarks used to train and evaluate vision-language models like CLIP, BLIP, and Flamingo.

Flickr30k is a **gated dataset** on HuggingFace. That means you need to:
1. Create a HuggingFace account at [huggingface.co](https://huggingface.co)
2. Go to [huggingface.co/datasets/nlphuji/flickr30k](https://huggingface.co/datasets/nlphuji/flickr30k) and accept the terms
3. Go to Settings > Access Tokens > create a new token (read access is enough)
4. In Colab, store it as a secret: click the key icon in the left sidebar, add `HF_TOKEN` with your token

This is the standard workflow you will use any time you need a gated model or dataset from HuggingFace -- whether it is Llama, Gemma, or any research dataset. Learn it once, use it everywhere.

In [ ]:
# ============================================================
# AUTHENTICATE WITH HUGGINGFACE
# ============================================================
# Option 1 (recommended): Use Colab secrets
# Click the key icon in the left sidebar, add HF_TOKEN
# Option 2: login() will prompt you to paste your token

from huggingface_hub import login

try:
    from google.colab import userdata          # only works in Colab
    HF_TOKEN = userdata.get('HF_TOKEN')        # read from Colab secrets
    login(token=HF_TOKEN)                      # authenticate silently
    print("Logged in via Colab secret.")
except Exception:
    login()  # fallback: interactive prompt, paste your token

In [ ]:
# ============================================================
# LOAD FLICKR30K FROM HUGGINGFACE
# ============================================================
from datasets import load_dataset

# Load the dataset -- this downloads images + captions
# We use the test split (has ~1K images) to keep it fast,
# or the train split with a subset for more variety
USE_FLICKR = True  # set to False to use pokemon fallback

if USE_FLICKR:
    try:
        hf_dataset = load_dataset("nlphuji/flickr30k", split="test")
        print(f"Loaded Flickr30k test split: {len(hf_dataset)} images")
        # Each item has: 'image' (PIL), 'caption' (list of 5 strings)
        print(f"Columns: {hf_dataset.column_names}")
        print(f"Example caption: {hf_dataset[0]['caption'][0]}")
    except Exception as e:
        print(f"Flickr30k failed ({e}), falling back to Pokemon...")
        USE_FLICKR = False

if not USE_FLICKR:
    # Fallback: pokemon-blip-captions (public, no auth needed)
    hf_dataset = load_dataset("lambdalabs/pokemon-blip-captions", split="train")
    print(f"Loaded Pokemon dataset: {len(hf_dataset)} images")

In [ ]:
# ============================================================
# PEEK AT THE DATA: show some images with their captions
# ============================================================

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    item = hf_dataset[i]
    img = item['image']                          # PIL image
    # Get caption (Flickr has list of 5, Pokemon has single string)
    if isinstance(item.get('caption', item.get('text', '')), list):
        caption = item['caption'][0]             # take first caption
    else:
        caption = item.get('caption', item.get('text', ''))
    ax.imshow(img)
    ax.set_title(caption[:50] + '...' if len(caption) > 50 else caption, fontsize=9)
    ax.axis('off')
plt.suptitle('Flickr30k: Real Photos with Human Captions', fontsize=14)
plt.tight_layout()
plt.show()

Now we need to wrap this into a PyTorch Dataset class. This is the standard pattern you will use for any custom dataset:

- `__len__` returns how many items are in the dataset
- `__getitem__` returns one (image, caption, clip_embedding) tuple

We will also **pre-compute CLIP embeddings** for every caption right now. That way during training we just look them up by index instead of running CLIP every batch. This is a common optimization -- CLIP is frozen, so its outputs never change.

In [ ]:
# ============================================================
# LOAD CLIP: we need it to encode captions into embeddings
# ============================================================
import open_clip

# ViT-B/32 is the standard CLIP model -- small, fast, good enough
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='laion2b_s34b_b79k'
)
clip_model = clip_model.to(device).eval()      # move to GPU, eval mode (no training)
tokenizer = open_clip.get_tokenizer('ViT-B-32')  # text tokenizer
CLIP_DIM = 512                                  # CLIP ViT-B/32 output dimension
print(f"CLIP loaded. Text embedding dim: {CLIP_DIM}")

In [ ]:
# ============================================================
# CUSTOM DATASET CLASS + PRE-COMPUTE CLIP EMBEDDINGS
# ============================================================

class CaptionedImageDataset(Dataset):
    """Wraps a HuggingFace image-caption dataset for PyTorch training.
    - Resizes all images to IMG_SIZE x IMG_SIZE
    - Stores pre-computed CLIP embeddings for fast training
    """
    def __init__(self, hf_dataset, clip_model, tokenizer, img_size=64, max_samples=5000):
        self.img_size = img_size

        # Image transform: resize to 64x64, convert to tensor [0,1]
        self.transform = T.Compose([
            T.Resize((img_size, img_size)),    # resize to fixed size
            T.ToTensor(),                      # PIL -> tensor, scales to [0,1]
        ])

        # Limit dataset size for fast training
        n = min(len(hf_dataset), max_samples)
        self.images = []     # will store PIL images
        self.captions = []   # will store caption strings

        print(f"Loading {n} images and captions...")
        for i in range(n):
            item = hf_dataset[i]
            img = item['image'].convert('RGB')   # ensure RGB (some images might be grayscale)
            # Get caption: Flickr has list, Pokemon has string
            cap = item.get('caption', item.get('text', ''))
            if isinstance(cap, list):
                cap = cap[0]                     # take first caption
            self.images.append(img)
            self.captions.append(cap)

        # Pre-compute ALL CLIP embeddings at once (batched, much faster)
        print("Pre-computing CLIP embeddings for all captions...")
        self.clip_embeddings = self._compute_clip_embeddings(clip_model, tokenizer)
        print(f"Dataset ready: {len(self)} images, CLIP embeddings shape: {self.clip_embeddings.shape}")

    def _compute_clip_embeddings(self, clip_model, tokenizer, batch_size=256):
        """Encode all captions with CLIP in batches."""
        all_embeddings = []
        with torch.no_grad():                    # no gradients needed
            for i in range(0, len(self.captions), batch_size):
                batch_captions = self.captions[i:i+batch_size]
                tokens = tokenizer(batch_captions).to(device)   # tokenize batch
                emb = clip_model.encode_text(tokens)             # encode to vectors
                emb = F.normalize(emb, dim=-1)                   # L2 normalize
                all_embeddings.append(emb.cpu())                 # store on CPU
        return torch.cat(all_embeddings, dim=0)  # shape: (N, 512)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.transform(self.images[idx])     # PIL -> (3, 64, 64) tensor
        clip_emb = self.clip_embeddings[idx]         # (512,) pre-computed
        return image, clip_emb


# Create dataset and dataloader
dataset = CaptionedImageDataset(
    hf_dataset, clip_model, tokenizer,
    img_size=IMG_SIZE,
    max_samples=5000     # 5K images is plenty for a demo
)

BATCH_SIZE = 64  # fits comfortably on T4 (16GB VRAM)
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,          # randomize order each epoch
    num_workers=2,         # parallel data loading
    drop_last=True         # drop incomplete last batch
)

# Show some training images
sample_images, sample_clip = next(iter(dataloader))
show_images(sample_images[:16], title="Training Images (resized to 64x64)", nrow=8)
print(f"Batch image shape: {sample_images.shape}")    # (64, 3, 64, 64)
print(f"Batch CLIP shape:  {sample_clip.shape}")      # (64, 512)

---
## Part 1: Convolutional VAE

Remember autoencoders from Day 11? Encoder compresses, decoder reconstructs. The problem was: the latent space had holes -- you could not sample random points and get real images out.

A VAE fixes this by making the encoder output a **probability distribution** (mean mu and variance sigma) instead of a fixed vector. We sample from that distribution, and a KL divergence loss pushes it to stay close to a standard normal N(0,1). That way, when we sample random points from N(0,1) at inference time, the decoder knows how to handle them.

Now we are doing this with **convolutional layers** instead of linear layers -- because images have spatial structure. The encoder uses Conv2d (downsamples), the decoder uses ConvTranspose2d (upsamples).

```
Encoder: (3,64,64) -> Conv(32,32,32) -> Conv(64,16,16) -> Conv(128,8,8) -> Conv(256,4,4) -> Flatten -> mu, logvar
Decoder: z -> Linear -> Reshape(256,4,4) -> ConvT(128,8,8) -> ConvT(64,16,16) -> ConvT(32,32,32) -> ConvT(3,64,64)
```

In [ ]:
# ============================================================
# CONVOLUTIONAL VAE
# ============================================================

class ConvVAE(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.latent_dim = latent_dim

        # --- Encoder: (3,64,64) -> (256,4,4) -> mu, logvar ---
        # Each Conv2d: kernel=4, stride=2, padding=1 halves spatial dims
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1),       # (3,64,64)   -> (32,32,32)
            nn.BatchNorm2d(32),               # normalize activations (stabilizes training)
            nn.ReLU(),                        # activation
            nn.Conv2d(32, 64, 4, 2, 1),      # (32,32,32)  -> (64,16,16)
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, 2, 1),     # (64,16,16)  -> (128,8,8)
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 256, 4, 2, 1),    # (128,8,8)   -> (256,4,4)
            nn.BatchNorm2d(256),
            nn.ReLU(),
        )
        # After encoder: flatten (256*4*4 = 4096) -> mu and logvar
        self.fc_mu = nn.Linear(256 * 4 * 4, latent_dim)      # mean of distribution
        self.fc_logvar = nn.Linear(256 * 4 * 4, latent_dim)  # log-variance

        # --- Decoder: z -> (256,4,4) -> (3,64,64) ---
        self.fc_decode = nn.Linear(latent_dim, 256 * 4 * 4)  # latent -> flat features
        # Each ConvTranspose2d: kernel=4, stride=2, padding=1 doubles spatial dims
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1),  # (256,4,4)  -> (128,8,8)
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1),   # (128,8,8)  -> (64,16,16)
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1),    # (64,16,16) -> (32,32,32)
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, 2, 1),     # (32,32,32) -> (3,64,64)
            nn.Sigmoid(),   # output in [0,1] to match image pixel range
        )

    def encode(self, x):
        """Image -> (mu, logvar)."""
        h = self.encoder(x)                  # (B, 256, 4, 4)
        h = h.view(h.size(0), -1)            # flatten to (B, 4096)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        """Sample z = mu + std * noise (the reparameterization trick)."""
        std = torch.exp(0.5 * logvar)        # std = exp(0.5 * log(var))
        eps = torch.randn_like(std)           # random noise ~ N(0,1)
        return mu + std * eps                 # differentiable sampling

    def decode(self, z):
        """Latent z -> reconstructed image."""
        h = self.fc_decode(z)                # (B, 4096)
        h = h.view(-1, 256, 4, 4)            # reshape to (B, 256, 4, 4)
        return self.decoder(h)               # (B, 3, 64, 64)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

In [ ]:
# ============================================================
# VAE LOSS: Reconstruction + KL Divergence
# ============================================================

def vae_loss(x_recon, x, mu, logvar):
    """VAE loss = reconstruction + KL divergence."""
    # Reconstruction: how well did we rebuild the image? (per-pixel MSE)
    recon = F.mse_loss(x_recon, x, reduction='sum') / x.size(0)
    # KL divergence: how far is q(z|x) from N(0,1)?
    # Closed form: -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    return recon + kl


# ============================================================
# TRAIN VAE (~15 epochs, ~2 min on T4)
# ============================================================

vae = ConvVAE(latent_dim=128).to(device)
vae_opt = optim.Adam(vae.parameters(), lr=1e-3)
vae_losses = []

vae.train()
for epoch in range(15):
    epoch_loss = 0
    for images, _ in tqdm(dataloader, desc=f"VAE {epoch+1}/15", leave=False):
        images = images.to(device)               # (B, 3, 64, 64)
        x_recon, mu, logvar = vae(images)        # forward pass
        loss = vae_loss(x_recon, images, mu, logvar)

        vae_opt.zero_grad()                      # clear old gradients
        loss.backward()                          # compute new gradients
        vae_opt.step()                           # update weights

        epoch_loss += loss.item()
        vae_losses.append(loss.item())
    print(f"VAE Epoch {epoch+1}/15  loss: {epoch_loss/len(dataloader):.1f}")

plot_loss(vae_losses, "VAE Training Loss")

In [ ]:
# ============================================================
# VAE GENERATION: sample z from N(0,1), pass through decoder
# ============================================================

vae.eval()
with torch.no_grad():
    # Random generation: sample latent vectors from standard normal
    z = torch.randn(16, 128).to(device)      # 16 random 128-dim vectors
    generated = vae.decode(z)                  # decode to images
show_images(generated, title="VAE: Random Samples from N(0,1)")

# Interpolation: walk smoothly between two latent points
with torch.no_grad():
    z1 = torch.randn(1, 128).to(device)       # start point
    z2 = torch.randn(1, 128).to(device)       # end point
    interps = []
    for t in np.linspace(0, 1, 8):             # 8 steps from z1 to z2
        z_t = (1 - t) * z1 + t * z2           # linear interpolation
        interps.append(vae.decode(z_t))
    interps = torch.cat(interps, dim=0)
show_images(interps, title="VAE: Latent Space Interpolation (z1 -> z2)", nrow=8)

---
## Part 2: DCGAN

Notice the VAE outputs are blurry. That is because MSE loss averages over all possible outputs -- if the true image could be a cat or a dog, MSE produces something in between (blur).

GANs take a completely different approach. Instead of comparing pixels, we train a second network -- the Discriminator -- to judge whether an image looks real or fake. The Generator learns to produce images that fool the Discriminator. They push each other to get better.

The Generator uses **transposed convolutions** (ConvTranspose2d) to go from a small noise vector to a full image. This is the inverse of a normal convolution: instead of shrinking, it expands. Each layer doubles the spatial resolution.

```
Generator: z(128) -> reshape(256,4,4) -> ConvT(128,8,8) -> ConvT(64,16,16) -> ConvT(32,32,32) -> ConvT(3,64,64)
Discriminator: (3,64,64) -> Conv(32,32,32) -> Conv(64,16,16) -> Conv(128,8,8) -> Conv(256,4,4) -> flatten -> real/fake
```

In [ ]:
# ============================================================
# DCGAN: Generator (noise -> image) + Discriminator (image -> real/fake)
# ============================================================

NOISE_DIM = 128  # dimension of random noise input to generator

class Generator(nn.Module):
    """Takes a noise vector z (128-dim) and produces a 64x64 RGB image."""
    def __init__(self, noise_dim=NOISE_DIM):
        super().__init__()
        # First: project noise to a small spatial feature map
        self.fc = nn.Linear(noise_dim, 256 * 4 * 4)  # noise -> flat features

        # Then: upsample with transposed convolutions
        # Each ConvTranspose2d doubles the spatial dimensions
        self.net = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1),  # (256,4,4)  -> (128,8,8)
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1),   # (128,8,8)  -> (64,16,16)
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1),    # (64,16,16) -> (32,32,32)
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, 2, 1),     # (32,32,32) -> (3,64,64)
            nn.Tanh(),  # output in [-1, 1] (standard for GANs)
        )

    def forward(self, z):
        h = self.fc(z)                       # (B, 4096)
        h = h.view(-1, 256, 4, 4)            # reshape to spatial (B, 256, 4, 4)
        return self.net(h)                   # upsample to (B, 3, 64, 64)


class Discriminator(nn.Module):
    """Takes a 64x64 RGB image and outputs probability it is real."""
    def __init__(self):
        super().__init__()
        # Mirror of generator: Conv2d halves spatial dims each layer
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1),       # (3,64,64)  -> (32,32,32)
            nn.LeakyReLU(0.2),                # leaky relu (better gradients for D)
            nn.Conv2d(32, 64, 4, 2, 1),      # (32,32,32) -> (64,16,16)
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, 2, 1),     # (64,16,16) -> (128,8,8)
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 4, 2, 1),    # (128,8,8)  -> (256,4,4)
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),
        )
        self.fc = nn.Linear(256 * 4 * 4, 1)  # flatten -> single real/fake score

    def forward(self, x):
        h = self.net(x)                      # (B, 256, 4, 4)
        h = h.view(h.size(0), -1)            # flatten to (B, 4096)
        return torch.sigmoid(self.fc(h))     # probability in [0,1]

In [ ]:
# ============================================================
# TRAIN GAN (~20 epochs, ~3 min on T4)
# ============================================================
# GAN training alternates: train D to tell real from fake, then train G to fool D
# Images must be in [-1,1] range for Tanh generator output

G = Generator().to(device)
D = Discriminator().to(device)
opt_G = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))  # standard GAN lr
opt_D = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
criterion = nn.BCELoss()  # binary cross entropy for real/fake

g_losses, d_losses = [], []

for epoch in range(20):
    for images, _ in tqdm(dataloader, desc=f"GAN {epoch+1}/20", leave=False):
        bs = images.size(0)
        # Scale images from [0,1] to [-1,1] to match Tanh output
        real = (images.to(device) * 2) - 1
        real_label = torch.ones(bs, 1, device=device)    # label 1 = real
        fake_label = torch.zeros(bs, 1, device=device)   # label 0 = fake

        # ----- Train Discriminator -----
        z = torch.randn(bs, NOISE_DIM, device=device)    # random noise
        fake = G(z).detach()                # generate fakes, detach so G is not updated
        loss_D = criterion(D(real), real_label) + criterion(D(fake), fake_label)

        opt_D.zero_grad()
        loss_D.backward()
        opt_D.step()

        # ----- Train Generator -----
        z = torch.randn(bs, NOISE_DIM, device=device)    # fresh noise
        fake = G(z)                                       # generate new fakes
        loss_G = criterion(D(fake), real_label)           # G wants D to say "real"

        opt_G.zero_grad()
        loss_G.backward()
        opt_G.step()

        g_losses.append(loss_G.item())
        d_losses.append(loss_D.item())

    print(f"GAN Epoch {epoch+1}/20  D: {np.mean(d_losses[-len(dataloader):]):.3f}  G: {np.mean(g_losses[-len(dataloader):]):.3f}")

# Plot both losses
plt.figure(figsize=(10, 3))
plt.plot(d_losses[::5], label='Discriminator', alpha=0.7)
plt.plot(g_losses[::5], label='Generator', alpha=0.7)
plt.legend(); plt.title('GAN Losses'); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# GAN GENERATION: sample noise, pass through Generator
# ============================================================

G.eval()
with torch.no_grad():
    z = torch.randn(16, NOISE_DIM, device=device)
    generated = G(z)
    generated = (generated + 1) / 2  # scale from [-1,1] back to [0,1] for display
show_images(generated, title="GAN: Generated Images (sharper than VAE!)")

---
## Part 3: Diffusion Model with U-Net

Now the best approach. Forget adversarial training -- diffusion models just add noise to images step by step, then train a network to predict what noise was added. The loss is plain MSE. No two-network balancing act, no mode collapse.

The network architecture is a **U-Net** -- the same architecture you saw in image segmentation on Day 11. It has a contracting path (downsample), a bottleneck, and an expanding path (upsample) with **skip connections** that carry detail from encoder to decoder.

One extra ingredient: the network needs to know **which timestep** it is denoising. We encode the timestep as a vector (like positional encoding in Transformers) and add it to the features inside the U-Net.

**Forward process:** gradually add noise over T steps until image becomes pure noise

**Reverse process:** start from noise, predict and remove noise step by step

**Training:** pick random t, add noise to that level, predict the noise, MSE loss

In [ ]:
# ============================================================
# DIFFUSION: Noise Schedule
# ============================================================
# These numbers control how much noise is added at each step

T = 500  # total diffusion steps (500 is enough for 64x64)

# Linear schedule: beta goes from very small to moderate
beta = torch.linspace(1e-4, 0.02, T).to(device)

# Pre-compute quantities we use repeatedly during training and sampling
alpha = 1.0 - beta                          # alpha_t = 1 - beta_t
alpha_bar = torch.cumprod(alpha, dim=0)     # cumulative product: how much signal remains at step t
sqrt_ab = torch.sqrt(alpha_bar)             # sqrt(alpha_bar_t) -- scales the clean image
sqrt_1_ab = torch.sqrt(1 - alpha_bar)       # sqrt(1-alpha_bar_t) -- scales the noise

print(f"t=0:   signal={alpha_bar[0]:.4f} (almost clean)")
print(f"t=250: signal={alpha_bar[250]:.4f} (half noise)")
print(f"t=499: signal={alpha_bar[499]:.4f} (almost pure noise)")

In [ ]:
# ============================================================
# VISUALIZE FORWARD PROCESS: watch an image turn to noise
# ============================================================

sample_img = sample_images[0:1].to(device)  # one training image
steps_to_show = [0, 50, 100, 200, 300, 400, 499]

noisy_imgs = []
for t_val in steps_to_show:
    noise = torch.randn_like(sample_img)
    # Forward process formula: x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * noise
    x_t = sqrt_ab[t_val] * sample_img + sqrt_1_ab[t_val] * noise
    noisy_imgs.append(x_t)

noisy_imgs = torch.cat(noisy_imgs, dim=0)
show_images(noisy_imgs, title=f"Forward Process: t={steps_to_show}", nrow=7)

In [ ]:
# ============================================================
# U-NET: The denoising network
# ============================================================
# Down blocks (encoder) -> Bottleneck -> Up blocks (decoder)
# Skip connections carry fine detail from encoder to decoder
# Time embedding tells the network which noise level it is at

class ConvBlock(nn.Module):
    """Two convolutions with batch norm and activation.
    Optionally adds a time/condition embedding to the features."""
    def __init__(self, in_ch, out_ch, emb_dim=128):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)   # same-size conv
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)  # same-size conv
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.emb_proj = nn.Linear(emb_dim, out_ch)  # project embedding to channel dim

    def forward(self, x, emb):
        h = F.relu(self.bn1(self.conv1(x)))          # first conv + norm + relu
        # Add time/condition embedding: project to (B, C), then add to every spatial position
        emb_out = self.emb_proj(emb)[:, :, None, None]  # (B, C) -> (B, C, 1, 1)
        h = h + emb_out                               # broadcast add across H, W
        h = F.relu(self.bn2(self.conv2(h)))           # second conv + norm + relu
        return h


class UNet(nn.Module):
    """Simple U-Net for diffusion denoising.
    Input: noisy image (B, 3, 64, 64) + timestep t
    Output: predicted noise (B, 3, 64, 64)
    """
    def __init__(self, emb_dim=128):
        super().__init__()

        # Time embedding: integer t -> learned vector
        self.time_mlp = nn.Sequential(
            nn.Linear(1, emb_dim),       # scalar -> emb_dim
            nn.SiLU(),                   # smooth activation
            nn.Linear(emb_dim, emb_dim), # refine
        )

        # --- Encoder (downsampling) ---
        self.down1 = ConvBlock(3, 64, emb_dim)       # (3,64,64)   -> (64,64,64)
        self.pool1 = nn.MaxPool2d(2)                  # (64,64,64)  -> (64,32,32)
        self.down2 = ConvBlock(64, 128, emb_dim)     # (64,32,32)  -> (128,32,32)
        self.pool2 = nn.MaxPool2d(2)                  # (128,32,32) -> (128,16,16)
        self.down3 = ConvBlock(128, 256, emb_dim)    # (128,16,16) -> (256,16,16)
        self.pool3 = nn.MaxPool2d(2)                  # (256,16,16) -> (256,8,8)

        # --- Bottleneck ---
        self.bottleneck = ConvBlock(256, 512, emb_dim)  # (256,8,8) -> (512,8,8)

        # --- Decoder (upsampling with skip connections) ---
        self.up3 = nn.ConvTranspose2d(512, 256, 2, 2)  # (512,8,8) -> (256,16,16)
        self.dec3 = ConvBlock(512, 256, emb_dim)       # concat skip: 256+256=512 -> 256
        self.up2 = nn.ConvTranspose2d(256, 128, 2, 2)  # (256,16,16) -> (128,32,32)
        self.dec2 = ConvBlock(256, 128, emb_dim)       # concat skip: 128+128=256 -> 128
        self.up1 = nn.ConvTranspose2d(128, 64, 2, 2)   # (128,32,32) -> (64,64,64)
        self.dec1 = ConvBlock(128, 64, emb_dim)        # concat skip: 64+64=128 -> 64

        # Final conv: map features to 3 channels (predicted noise)
        self.final = nn.Conv2d(64, 3, 1)               # (64,64,64) -> (3,64,64)

    def forward(self, x, t):
        # Compute time embedding
        t_emb = self.time_mlp(t.float().unsqueeze(-1) / T)  # normalize t to [0,1]

        # Encoder
        d1 = self.down1(x, t_emb)            # (B,64,64,64)  -- save for skip
        d2 = self.down2(self.pool1(d1), t_emb)  # (B,128,32,32) -- save for skip
        d3 = self.down3(self.pool2(d2), t_emb)  # (B,256,16,16) -- save for skip

        # Bottleneck
        b = self.bottleneck(self.pool3(d3), t_emb)  # (B,512,8,8)

        # Decoder with skip connections (concatenate encoder features)
        u3 = self.up3(b)                     # (B,256,16,16)
        u3 = self.dec3(torch.cat([u3, d3], dim=1), t_emb)  # skip: cat -> (B,512,16,16) -> (B,256,16,16)
        u2 = self.up2(u3)                    # (B,128,32,32)
        u2 = self.dec2(torch.cat([u2, d2], dim=1), t_emb)  # skip: cat -> (B,256,32,32) -> (B,128,32,32)
        u1 = self.up1(u2)                    # (B,64,64,64)
        u1 = self.dec1(torch.cat([u1, d1], dim=1), t_emb)  # skip: cat -> (B,128,64,64) -> (B,64,64,64)

        return self.final(u1)                # (B,3,64,64) predicted noise


# Quick check: does the shape work?
test_model = UNet().to(device)
test_x = torch.randn(2, 3, 64, 64, device=device)
test_t = torch.tensor([10, 200], device=device)
test_out = test_model(test_x, test_t)
print(f"U-Net input: {test_x.shape} -> output: {test_out.shape}")  # should match
print(f"U-Net parameters: {sum(p.numel() for p in test_model.parameters()):,}")
del test_model, test_x, test_t, test_out  # free memory

In [ ]:
# ============================================================
# TRAIN DIFFUSION MODEL (~15 epochs, ~4 min on T4)
# ============================================================
# Training is embarrassingly simple:
# 1. Pick random timestep t
# 2. Add noise to that level
# 3. Predict the noise
# 4. MSE loss between predicted and actual noise

unet = UNet().to(device)
unet_opt = optim.Adam(unet.parameters(), lr=1e-3)
diff_losses = []

unet.train()
for epoch in range(15):
    epoch_loss = 0
    for images, _ in tqdm(dataloader, desc=f"Diff {epoch+1}/15", leave=False):
        images = images.to(device)                    # (B, 3, 64, 64)

        # Step 1: random timestep for each image in batch
        t = torch.randint(0, T, (images.size(0),), device=device)

        # Step 2: sample random noise
        noise = torch.randn_like(images)

        # Step 3: create noisy image at timestep t
        # x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * noise
        x_t = sqrt_ab[t, None, None, None] * images + sqrt_1_ab[t, None, None, None] * noise

        # Step 4: predict the noise
        predicted_noise = unet(x_t, t)

        # Step 5: simple MSE loss
        loss = F.mse_loss(predicted_noise, noise)

        unet_opt.zero_grad()
        loss.backward()
        unet_opt.step()

        epoch_loss += loss.item()
        diff_losses.append(loss.item())

    print(f"Diff Epoch {epoch+1}/15  loss: {epoch_loss/len(dataloader):.4f}")

plot_loss(diff_losses, "Diffusion Training Loss (MSE on noise prediction)")

In [ ]:
# ============================================================
# DIFFUSION SAMPLING: start from noise, denoise step by step
# ============================================================

@torch.no_grad()
def sample_diffusion(model, n_samples=16):
    """Generate images by iterative denoising."""
    model.eval()
    # Start from pure random noise
    x = torch.randn(n_samples, 3, IMG_SIZE, IMG_SIZE, device=device)

    # Denoise step by step: t = T-1, T-2, ..., 0
    for t_val in tqdm(reversed(range(T)), total=T, desc="Sampling", leave=False):
        t_batch = torch.full((n_samples,), t_val, device=device)
        predicted_noise = model(x, t_batch)     # predict noise at this step

        # Reverse step formula
        alpha_t = alpha[t_val]
        alpha_bar_t = alpha_bar[t_val]
        beta_t = beta[t_val]

        x = (1 / torch.sqrt(alpha_t)) * (
            x - (beta_t / torch.sqrt(1 - alpha_bar_t)) * predicted_noise
        )

        if t_val > 0:  # add stochastic noise (except at last step)
            x = x + torch.sqrt(beta_t) * torch.randn_like(x)

    return x.clamp(0, 1)


generated = sample_diffusion(unet, n_samples=16)
show_images(generated, title="Diffusion Model: Generated Images")

---
## Part 4: CLIP-Conditioned Diffusion

Everything so far generates random images -- we have no control over *what* gets generated. To fix that, we need to tell the model what we want using text.

We already have CLIP embeddings for every caption in our dataset. The idea is simple: take the CLIP text embedding (512-dim vector), project it down to 128 dimensions, and **add it to the time embedding**. Now the U-Net knows both *which noise level* it is at AND *what the image should depict*.

This is exactly the mechanism behind Stable Diffusion, just simplified. In Stable Diffusion:
- CLIP encodes the text prompt
- The embedding is injected into the U-Net via cross-attention
- The U-Net denoises conditioned on the text

We are doing the same thing, but using addition instead of cross-attention (simpler, works well for a demo). The principle is identical: the text embedding steers the denoising process.

In [ ]:
# ============================================================
# CONDITIONAL U-NET: same architecture + CLIP condition
# ============================================================
# Only change: we add a condition projection that maps CLIP (512-dim) to emb_dim,
# then add it to the time embedding. Everything else is identical.

class ConditionalUNet(nn.Module):
    def __init__(self, emb_dim=128, clip_dim=512):
        super().__init__()

        # Time embedding (same as before)
        self.time_mlp = nn.Sequential(
            nn.Linear(1, emb_dim),
            nn.SiLU(),
            nn.Linear(emb_dim, emb_dim),
        )

        # NEW: Condition projection -- maps CLIP embedding to same space as time
        self.cond_proj = nn.Sequential(
            nn.Linear(clip_dim, emb_dim),    # 512 -> 128
            nn.SiLU(),
            nn.Linear(emb_dim, emb_dim),     # 128 -> 128
        )

        # Encoder
        self.down1 = ConvBlock(3, 64, emb_dim)
        self.pool1 = nn.MaxPool2d(2)
        self.down2 = ConvBlock(64, 128, emb_dim)
        self.pool2 = nn.MaxPool2d(2)
        self.down3 = ConvBlock(128, 256, emb_dim)
        self.pool3 = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = ConvBlock(256, 512, emb_dim)

        # Decoder
        self.up3 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dec3 = ConvBlock(512, 256, emb_dim)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec2 = ConvBlock(256, 128, emb_dim)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec1 = ConvBlock(128, 64, emb_dim)
        self.final = nn.Conv2d(64, 3, 1)

    def forward(self, x, t, cond):
        """
        x: noisy image (B, 3, 64, 64)
        t: timestep (B,)
        cond: CLIP text embedding (B, 512)
        """
        # Combine time + condition into one embedding
        t_emb = self.time_mlp(t.float().unsqueeze(-1) / T)  # (B, 128)
        c_emb = self.cond_proj(cond)                         # (B, 128)
        emb = t_emb + c_emb  # simply ADD them -- the model learns to use both

        # Same U-Net forward pass, using combined embedding
        d1 = self.down1(x, emb)
        d2 = self.down2(self.pool1(d1), emb)
        d3 = self.down3(self.pool2(d2), emb)

        b = self.bottleneck(self.pool3(d3), emb)

        u3 = self.dec3(torch.cat([self.up3(b), d3], dim=1), emb)
        u2 = self.dec2(torch.cat([self.up2(u3), d2], dim=1), emb)
        u1 = self.dec1(torch.cat([self.up1(u2), d1], dim=1), emb)

        return self.final(u1)

In [ ]:
# ============================================================
# TRAIN CONDITIONAL DIFFUSION (~15 epochs, ~4 min on T4)
# ============================================================
# Same training loop as before, but now we pass the CLIP embedding as condition

cond_unet = ConditionalUNet().to(device)
cond_opt = optim.Adam(cond_unet.parameters(), lr=1e-3)
cond_losses = []

cond_unet.train()
for epoch in range(15):
    epoch_loss = 0
    for images, clip_emb in tqdm(dataloader, desc=f"CondDiff {epoch+1}/15", leave=False):
        images = images.to(device)
        clip_emb = clip_emb.to(device)                # (B, 512) pre-computed CLIP embedding

        t = torch.randint(0, T, (images.size(0),), device=device)
        noise = torch.randn_like(images)
        x_t = sqrt_ab[t, None, None, None] * images + sqrt_1_ab[t, None, None, None] * noise

        # Only difference: pass CLIP embedding as condition
        predicted_noise = cond_unet(x_t, t, clip_emb)
        loss = F.mse_loss(predicted_noise, noise)

        cond_opt.zero_grad()
        loss.backward()
        cond_opt.step()

        epoch_loss += loss.item()
        cond_losses.append(loss.item())

    print(f"CondDiff Epoch {epoch+1}/15  loss: {epoch_loss/len(dataloader):.4f}")

plot_loss(cond_losses, "Conditional Diffusion Training Loss")

In [ ]:
# ============================================================
# TEXT-TO-IMAGE GENERATION: type a prompt, get an image
# ============================================================

@torch.no_grad()
def generate_from_text(model, text_prompt, n_samples=8):
    """Generate images conditioned on a text prompt."""
    model.eval()

    # Step 1: Encode text with CLIP (same as we did for captions)
    tokens = tokenizer([text_prompt]).to(device)
    text_emb = clip_model.encode_text(tokens)
    text_emb = F.normalize(text_emb, dim=-1)      # L2 normalize
    cond = text_emb.repeat(n_samples, 1)           # repeat for all samples

    # Step 2: Start from pure noise
    x = torch.randn(n_samples, 3, IMG_SIZE, IMG_SIZE, device=device)

    # Step 3: Denoise with condition
    for t_val in tqdm(reversed(range(T)), total=T, desc="Generating", leave=False):
        t_batch = torch.full((n_samples,), t_val, device=device)
        predicted_noise = cond_unet(x, t_batch, cond)  # condition on text!

        alpha_t = alpha[t_val]
        alpha_bar_t = alpha_bar[t_val]
        beta_t = beta[t_val]

        x = (1 / torch.sqrt(alpha_t)) * (
            x - (beta_t / torch.sqrt(1 - alpha_bar_t)) * predicted_noise
        )
        if t_val > 0:
            x = x + torch.sqrt(beta_t) * torch.randn_like(x)

    return x.clamp(0, 1)


# Try different prompts!
prompts = [
    "a dog playing in the park",
    "a group of people on the beach",
    "a child riding a bicycle",
]

for prompt in prompts:
    generated = generate_from_text(cond_unet, prompt, n_samples=8)
    show_images(generated, title=f'Prompt: "{prompt}"', nrow=8)

---
## Part 5: Comparison

Let us put all four models side by side so you can see the progression: real images, VAE (blurry), GAN (sharper), diffusion (best), and text-conditioned diffusion (controllable). This is the history of generative models in one notebook.

In [ ]:
# ============================================================
# SIDE-BY-SIDE COMPARISON
# ============================================================

fig, axes = plt.subplots(1, 5, figsize=(25, 5))

# 1. Real images
real_grid = torchvision.utils.make_grid(sample_images[:8].cpu(), nrow=4, padding=2)
axes[0].imshow(real_grid.permute(1, 2, 0).clamp(0, 1).numpy())
axes[0].set_title('Real (Flickr30k)', fontsize=12)
axes[0].axis('off')

# 2. VAE
vae.eval()
with torch.no_grad():
    vae_imgs = vae.decode(torch.randn(8, 128, device=device)).cpu()
vae_grid = torchvision.utils.make_grid(vae_imgs.clamp(0, 1), nrow=4, padding=2)
axes[1].imshow(vae_grid.permute(1, 2, 0).numpy())
axes[1].set_title('VAE (blurry)', fontsize=12)
axes[1].axis('off')

# 3. GAN
G.eval()
with torch.no_grad():
    gan_imgs = ((G(torch.randn(8, NOISE_DIM, device=device)) + 1) / 2).cpu()
gan_grid = torchvision.utils.make_grid(gan_imgs.clamp(0, 1), nrow=4, padding=2)
axes[2].imshow(gan_grid.permute(1, 2, 0).numpy())
axes[2].set_title('GAN (sharper)', fontsize=12)
axes[2].axis('off')

# 4. Diffusion (unconditional)
diff_imgs = sample_diffusion(unet, n_samples=8).cpu()
diff_grid = torchvision.utils.make_grid(diff_imgs.clamp(0, 1), nrow=4, padding=2)
axes[3].imshow(diff_grid.permute(1, 2, 0).numpy())
axes[3].set_title('Diffusion (best)', fontsize=12)
axes[3].axis('off')

# 5. Conditional diffusion
cond_imgs = generate_from_text(cond_unet, "a person walking a dog", n_samples=8).cpu()
cond_grid = torchvision.utils.make_grid(cond_imgs.clamp(0, 1), nrow=4, padding=2)
axes[4].imshow(cond_grid.permute(1, 2, 0).numpy())
axes[4].set_title('CLIP-Conditioned\n"a person walking a dog"', fontsize=11)
axes[4].axis('off')

plt.suptitle('The Evolution: VAE -> GAN -> Diffusion -> Text-Conditioned', fontsize=14)
plt.tight_layout()
plt.show()

---
## Wrap-up

What you just saw is exactly how Stable Diffusion works, just at 64x64 instead of 512x512. The real system has three pieces:

1. **A VAE** compresses images to a smaller latent space (we built one)
2. **A U-Net** denoises in that latent space, conditioned on text (we built one)
3. **CLIP** provides the text understanding (we used it for conditioning)

Stable Diffusion runs diffusion in the VAE's latent space (64x64 latent instead of 512x512 pixels) and uses cross-attention instead of simple addition for the text conditioning. But the core ideas are identical to what we coded today.

| Model | Training | Quality | Speed | Control |
|-------|----------|---------|-------|---------|
| VAE | Stable (1 network, KL+recon loss) | Blurry | Fast (1 pass) | Latent z |
| GAN | Tricky (2 networks, adversarial) | Sharp | Fast (1 pass) | Noise z |
| Diffusion | Stable (1 network, MSE loss) | Best | Slow (T steps) | Noise z |
| + CLIP | Same as diffusion | Best | Slow (T steps) | Text! |

Tomorrow in the lab you will use a pre-trained Stable Diffusion model and see what happens when you scale this up to real resolution with billions of training images.